# kepler_rectilinear — a real bug this test suite caught

Calls the Kepler solver with (near-)rectilinear hyperbolic motion — the regime that strains its iteration hardest. This probe found a genuine port defect: with a large timestep the Rust looped forever where the C returned, because `while (a > b)` and `if (a <= b) break` differ when a value is NaN. Now fixed; the two agree bit-for-bit.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd rebound_rust
cargo build --release --example kepler_rectilinear
cd porttest
../target/release/examples/kepler_rectilinear 1e-12 1 2.0
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "rebound_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "kepler_rectilinear"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.00s


In [2]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + EXE)
res = subprocess.run([exe, "1e-12", "1", "2.0"], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

BEFORE: 20 kepler steps, vy=9.99999999999999980e-13
AFTER: x=401c000000000000 y=3d819799812dea11 vx=4008000000000000 vy=3d719799812dea11



In [3]:
print("(see the program output above)")

(see the program output above)
